## 1. Imports and paths

In [15]:
from pathlib import Path
import numpy as np
import pandas as pd

from pyBKT.models import Model

BASE_DIR = Path("..").resolve()          # app/
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = DATA_DIR / "outputs"
MODELS_DIR = BASE_DIR / "models"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_EVALUATIONS_XLSX = OUTPUTS_DIR / "bkt_model_evaluations_v.0.xlsx"
CHOSEN_MODEL_PROBABILITIES_XLSX = OUTPUTS_DIR / "bkt_chosen_model_probabilities_v.0.xlsx"

print("BASE_DIR:", BASE_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("MODEL_EVALUATIONS_XLSX:", MODEL_EVALUATIONS_XLSX)
print("CHOSEN_MODEL_PROBABILITIES_XLSX:", CHOSEN_MODEL_PROBABILITIES_XLSX)


BASE_DIR: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app
PROCESSED_DIR: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/data/processed
OUTPUTS_DIR: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/data/outputs
MODEL_EVALUATIONS_XLSX: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/data/outputs/bkt_model_evaluations_v.0.xlsx
CHOSEN_MODEL_PROBABILITIES_XLSX: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/data/outputs/bkt_chosen_model_probabilities_v.0.xlsx


## 2. Load cleaned dataset

In [16]:
TRAIN_06_PATH = PROCESSED_DIR / "df_train_kc_cleaned_2006-2007_v.0.csv"
TRAIN_08_PATH = PROCESSED_DIR / "df_train_kc_cleaned_2008-2009_v.0.csv"

df_06 = pd.read_csv(TRAIN_06_PATH, encoding="latin")
df_08 = pd.read_csv(TRAIN_08_PATH, encoding="latin")

print("2006-07 shape:", df_06.shape)
print("2008-09 shape:", df_08.shape)

print(df_06.columns.tolist())

2006-07 shape: (333293, 10)
2008-09 shape: (791162, 11)
['Anon Student Id', 'Problem Name', 'Step Name', 'Step Start Time', 'Step End Time', 'Correct First Attempt', 'Incorrects', 'Hints', 'Corrects', 'skill_name']


## 3. Helpers and mapping

In [17]:
KC_NAME_MAP = {
    "Expand / Eliminate Parentheses": "expand_eliminate_parentheses",
    "Combine Like Terms": "combine_like_terms",
    "Move Constants Across the Equation": "move_constants",
    "Move Variable Terms to One Side": "move_variables_one_side",
    "Remove Coefficient": "remove_coefficient",
    "Normalize Negative Variable / Sign": "normalize_negative_sign",
}

COLUMNS_MAP = {
    "Anon Student Id": "user_id",
    "skill_name": "skill_name",
    "Correct First Attempt": "correct",
    "Problem Name": "problem_id",
    "Move Variable Terms to One Side": "move_variables_one_side",
    "Remove Coefficient": "remove_coefficient",
    "Normalize Negative Variable / Sign": "normalize_negative_sign",
}

BASE_DEFAULTS = {
    "order_id": "order_id",
    "user_id": "user_id",
    "skill_name": "skill_name",
    "correct": "correct",
}

In [18]:
def prepare_columns(df):
    df = df.copy()
    time_cols = [
        "Step Start Time",
    ]
    for col in time_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    
    numeric_cols = [
        "Correct First Attempt",
        "Incorrects",
        "Hints",
        "Corrects"
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


## 4. Build the dataframe for pyBKT

This function transforms the cleaned datasets into the format expected by pyBKT.

In [19]:
def build_pybkt_df(df,dataset_tag,min_seq_len=2):
    data = df.copy()

    required_cols = [
        "Anon Student Id",
        "skill_name",
        "Correct First Attempt",
        "Step Start Time"
    ]
    missing = [c for c in required_cols if c not in data.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    data = prepare_columns(data)

    # Keep only valid rows
    data = data.dropna(subset=[
        "Anon Student Id",
        "skill_name",
        "Correct First Attempt",
        "Step Start Time"
    ]).copy()

    data["Correct First Attempt"] = pd.to_numeric(data["Correct First Attempt"], errors="coerce"
    )
    data = data[data["Correct First Attempt"].isin([0, 1])].copy()

    # Keep only the final selected KCs
    data = data[data["skill_name"].isin(KC_NAME_MAP.keys())].copy()

    # Stable IDs
    data["user_id"] = dataset_tag + "__" + data["Anon Student Id"].astype(str)
    data["skill_name"] = data["skill_name"].map(KC_NAME_MAP)
    data["correct"] = data["Correct First Attempt"].astype(int)

    sort_cols = ["user_id", "skill_name", "Step Start Time"]
    if "Row" in data.columns:
        sort_cols.append("Row")

    data = data.sort_values(sort_cols).reset_index(drop=True)

    # Sequential order_id within each student-skill sequence
    data["order_id"] = data.groupby(["user_id", "skill_name"]).cumcount() + 1

    # Remove sequences that are too short
    seq_sizes = (
        data.groupby(["user_id", "skill_name"])
        .size()
        .reset_index(name="n_obs")
    )
    keep_seq = seq_sizes[seq_sizes["n_obs"] >= min_seq_len][["user_id", "skill_name"]]

    data = data.merge(keep_seq, on=["user_id", "skill_name"], how="inner")

    keep_cols = ["order_id","user_id","skill_name","correct"]

    return data[keep_cols].reset_index(drop=True)

## 5. Create the three modelling datasets

The notebook evaluates:

- `2006-07`
- `2008-09`
- `2006-09`, created by concatenating both datasets

In [20]:
bkt_06 = build_pybkt_df(df_06, dataset_tag="2006_07")
bkt_08 = build_pybkt_df(df_08, dataset_tag="2008_09")
bkt_06_09 = pd.concat([bkt_06, bkt_08], ignore_index=True)

datasets_bkt = {
    "2006-07": bkt_06,
    "2008-09": bkt_08,
    "2006-09": bkt_06_09,
}

for name, df_ in datasets_bkt.items():
    print(
        f"{name}: rows={len(df_)}, students={df_['user_id'].nunique()}, "
        f"skills={df_['skill_name'].nunique()}"
    )

2006-07: rows=333209, students=1014, skills=5
2008-09: rows=775256, students=2241, skills=5
2006-09: rows=1108465, students=3255, skills=5


 ## 6. Define models 
 Only these two models are tested:

- `basic`
- `forgets`

Both return 5 probabilities per KC:

- `prior`
- `learns`
- `guesses`
- `slips`
- `forgets`

In the `basic` model, `forgets` is fixed at 0.

In [21]:
MODEL_SPECS = {
    "basic": {
        "defaults_extra": {},
        "fit_kwargs": {}
    },
    "forgets": {
        "defaults_extra": {},
        "fit_kwargs": {"forgets": True}
    },
}

## 7. Evaluation functions

In [22]:
def summarize_cv_output(cv_output):
    """
    pyBKT crossvalidate can return a scalar, list, array, Series or DataFrame.
    This function summarises the output using mean and standard deviation.
    """
    if isinstance(cv_output, pd.DataFrame):
        numeric = cv_output.select_dtypes(include=[np.number])
        arr = numeric.to_numpy().ravel()
    elif isinstance(cv_output, (list, tuple, np.ndarray, pd.Series)):
        arr = np.array(cv_output, dtype=float).ravel()
    else:
        arr = np.array([cv_output], dtype=float)

    arr = arr[~np.isnan(arr)]

    if len(arr) == 0:
        return np.nan, np.nan

    return float(np.mean(arr)), float(np.std(arr))


def normalise_bkt_params(raw_params, dataset_name, model_name):
    params = raw_params.copy().reset_index()

    rename_map = {}
    if "skill" in params.columns:
        rename_map["skill"] = "skill_name"
    if "param" in params.columns:
        rename_map["param"] = "parameter"
    if "value" in params.columns:
        rename_map["value"] = "probability"

    params = params.rename(columns=rename_map)

    unnamed_cols = [c for c in params.columns if str(c).startswith("level_")]
    if "skill_name" not in params.columns and len(unnamed_cols) >= 1:
        params = params.rename(columns={unnamed_cols[0]: "skill_name"})
    if "parameter" not in params.columns and len(unnamed_cols) >= 2:
        params = params.rename(columns={unnamed_cols[1]: "parameter"})
    if "class" not in params.columns and len(unnamed_cols) >= 3:
        params = params.rename(columns={unnamed_cols[2]: "class"})

    params["dataset_name"] = dataset_name
    params["model_name"] = model_name

    preferred_first = ["dataset_name", "model_name", "skill_name", "parameter", "class", "probability"]
    ordered_cols = [c for c in preferred_first if c in params.columns]
    ordered_cols += [c for c in params.columns if c not in ordered_cols]
    return params[ordered_cols]


def make_kc_probability_table(params_long):
    value_col = "probability" if "probability" in params_long.columns else params_long.select_dtypes(include=[np.number]).columns[-1]

    params_wide = (
        params_long.pivot_table(
            index=["dataset_name", "model_name", "skill_name"],
            columns="parameter",
            values=value_col,
            aggfunc="first"
        )
        .reset_index()
    )

    expected_probability_cols = ["prior", "learns", "guesses", "slips", "forgets"]
    for col in expected_probability_cols:
        if col not in params_wide.columns:
            params_wide[col] = 0.0 if col == "forgets" else np.nan

    ordered_cols = ["dataset_name", "model_name", "skill_name"] + expected_probability_cols
    ordered_cols += [c for c in params_wide.columns if c not in ordered_cols]
    return params_wide[ordered_cols]


def safe_sheet_name(sheet_name, used_names):
    invalid_chars = "[]:*?/\\"
    cleaned = ''.join('_' if ch in invalid_chars else ch for ch in str(sheet_name)).strip()
    cleaned = cleaned or "Sheet"
    cleaned = cleaned[:31]
    candidate = cleaned
    counter = 1

    while candidate in used_names:
        suffix = f"_{counter}"
        candidate = cleaned[:31 - len(suffix)] + suffix
        counter += 1

    used_names.add(candidate)
    return candidate


def save_sheets_to_excel(path, sheets):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    used_names = set()
    wrote_sheet = False

    with pd.ExcelWriter(path) as writer:
        for sheet_name, table in sheets.items():
            if table is None:
                continue

            if isinstance(table, pd.Series):
                table_df = table.to_frame().T
            elif isinstance(table, pd.DataFrame):
                table_df = table.copy()
            else:
                table_df = pd.DataFrame(table)

            if table_df.empty:
                table_df = pd.DataFrame({"message": ["No data available"]})

            table_df.to_excel(
                writer,
                sheet_name=safe_sheet_name(sheet_name, used_names),
                index=False
            )
            wrote_sheet = True

        if not wrote_sheet:
            pd.DataFrame({"message": ["No data available"]}).to_excel(
                writer,
                sheet_name="Summary",
                index=False
            )

    print(f"Saved Excel file: {path}")


def run_one_model_on_one_dataset(
    df,
    dataset_name,
    model_name,
    model_spec,
    folds=3,
    seed=42,
    num_fits=1
):
    defaults = BASE_DEFAULTS.copy()
    defaults.update(model_spec["defaults_extra"])

    fit_kwargs = model_spec["fit_kwargs"].copy()

    row = {
        "dataset_name": dataset_name,
        "model_name": model_name,
        "n_rows": len(df),
        "n_students": df["user_id"].nunique(),
        "n_skills": df["skill_name"].nunique(),
    }

    model = Model(seed=seed, num_fits=num_fits)
    model.fit(data=df, defaults=defaults, **fit_kwargs)

    params = normalise_bkt_params(
        raw_params=model.params(),
        dataset_name=dataset_name,
        model_name=model_name
    )

    train_preds = model.predict(data=df)
    train_rmse = model.evaluate(data=df, metric="rmse")
    train_auc = model.evaluate(data=df, metric="auc")
    train_acc = model.evaluate(data=df, metric="accuracy")

    cv_model_auc = Model(seed=seed, num_fits=num_fits)
    cv_auc_raw = cv_model_auc.crossvalidate(
        data=df,
        defaults=defaults,
        folds=folds,
        metric="auc",
        **fit_kwargs
    )
    cv_auc_mean, cv_auc_std = summarize_cv_output(cv_auc_raw)

    cv_model_rmse = Model(seed=seed, num_fits=num_fits)
    cv_rmse_raw = cv_model_rmse.crossvalidate(
        data=df,
        defaults=defaults,
        folds=folds,
        metric="rmse",
        **fit_kwargs
    )
    cv_rmse_mean, cv_rmse_std = summarize_cv_output(cv_rmse_raw)

    cv_model_acc = Model(seed=seed, num_fits=num_fits)
    cv_acc_raw = cv_model_acc.crossvalidate(
        data=df,
        defaults=defaults,
        folds=folds,
        metric="accuracy",
        **fit_kwargs
    )
    cv_acc_mean, cv_acc_std = summarize_cv_output(cv_acc_raw)

    row.update({
        "status": "ok",
        "train_auc": float(train_auc),
        "train_rmse": float(train_rmse),
        "train_accuracy": float(train_acc),
        "cv_auc_mean": cv_auc_mean,
        "cv_auc_std": cv_auc_std,
        "cv_rmse_mean": cv_rmse_mean,
        "cv_rmse_std": cv_rmse_std,
        "cv_accuracy_mean": cv_acc_mean,
        "cv_accuracy_std": cv_acc_std,
        "n_train_predictions": len(train_preds),
    })

    return row, params, train_preds


## 9. Run the comparison

In [23]:
def run_model_selection(
    datasets_dict,
    model_specs,
    folds=3,
    seed=42,
    num_fits=1
):
    results = []
    params_tables = []
    preds_tables = {}

    for dataset_name, df in datasets_dict.items():
        print("DATASET:", dataset_name)

        for model_name, model_spec in model_specs.items():
            print(f"Running {model_name}...")

            res_row, params_df, train_preds = run_one_model_on_one_dataset(
                df=df,
                dataset_name=dataset_name,
                model_name=model_name,
                model_spec=model_spec,
                folds=folds,
                seed=seed,
                num_fits=num_fits
            )

            results.append(res_row)

            if params_df is not None:
                params_tables.append(params_df)

            if train_preds is not None:
                preds_tables[(dataset_name, model_name)] = train_preds

    results_df = pd.DataFrame(results)

    if params_tables:
        params_df = pd.concat(params_tables, ignore_index=True)
    else:
        params_df = pd.DataFrame()

    return results_df, params_df, preds_tables

In [24]:
results_df, params_df, preds_tables = run_model_selection(
    datasets_dict=datasets_bkt,
    model_specs=MODEL_SPECS,
    folds=3,
    seed=42,
    num_fits=1
)

results_df

DATASET: 2006-07
Running basic...
Running forgets...
DATASET: 2008-09
Running basic...
Running forgets...
DATASET: 2006-09
Running basic...
Running forgets...


,dataset_name,model_name,n_rows,n_students,n_skills,status,train_auc,train_rmse,train_accuracy,cv_auc_mean,cv_auc_std,cv_rmse_mean,cv_rmse_std,cv_accuracy_mean,cv_accuracy_std,n_train_predictions
0,2006-07,basic,333209,1014,5,ok,0.66739,0.34620,0.84951,0.63804,0.05591,0.36269,0.04104,0.82809,0.05078,333209
1,2006-07,forgets,333209,1014,5,ok,0.71478,0.34123,0.85036,0.68575,0.04573,0.35934,0.04417,0.82834,0.05079,333209
2,2008-09,basic,775256,2241,5,ok,0.67526,0.30602,0.88724,0.64035,0.05857,0.32817,0.05138,0.85909,0.06433,775256
3,2008-09,forgets,775256,2241,5,ok,0.71902,0.30183,0.88724,0.67932,0.02554,0.32371,0.05183,0.85909,0.06433,775256
4,2006-09,basic,1108465,3255,5,ok,0.66891,0.31975,0.87456,0.65578,0.04594,0.33926,0.04588,0.84836,0.06002,1108465
5,2006-09,forgets,1108465,3255,5,ok,0.71925,0.31465,0.87456,0.68430,0.04298,0.33571,0.04663,0.84836,0.06002,1108465


## 10. Select the best model

Selection criteria:

1. Highest `cv_auc_mean`
2. Lowest `cv_rmse_mean`
3. If tied, choose the simpler model

In [25]:
MODEL_COMPLEXITY = {
    "basic": 0,
    "forgets": 1,
}

ranked_results = results_df.copy()
ranked_results = ranked_results[ranked_results["status"] == "ok"].copy()

ranked_results["model_complexity"] = ranked_results["model_name"].map(MODEL_COMPLEXITY)

ranked_results = ranked_results.sort_values(
    by=["dataset_name", "cv_auc_mean", "cv_rmse_mean", "model_complexity"],
    ascending=[True, False, True, True]
).reset_index(drop=True)

best_models = (
    ranked_results.groupby("dataset_name", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("BEST MODEL PER DATASET")
display(best_models[[
    "dataset_name",
    "model_name",
    "train_auc",
    "train_rmse",
    "train_accuracy",
    "cv_auc_mean",
    "cv_rmse_mean",
    "cv_accuracy_mean"
]])

print("FULL RANKING")
display(ranked_results[[
    "dataset_name",
    "model_name",
    "train_auc",
    "train_rmse",
    "train_accuracy",
    "cv_auc_mean",
    "cv_rmse_mean",
    "cv_accuracy_mean",
    "model_complexity"
]])

save_sheets_to_excel(
    MODEL_EVALUATIONS_XLSX,
    {
        "model_evaluations": results_df,
        "full_ranking": ranked_results,
        "best_model_per_dataset": best_models,
        "learned_parameters_long": params_df,
    }
)


BEST MODEL PER DATASET


,dataset_name,model_name,train_auc,train_rmse,train_accuracy,cv_auc_mean,cv_rmse_mean,cv_accuracy_mean
0,2006-07,forgets,0.71478,0.34123,0.85036,0.68575,0.35934,0.82834
1,2006-09,forgets,0.71925,0.31465,0.87456,0.68430,0.33571,0.84836
2,2008-09,forgets,0.71902,0.30183,0.88724,0.67932,0.32371,0.85909


FULL RANKING


,dataset_name,model_name,train_auc,train_rmse,train_accuracy,cv_auc_mean,cv_rmse_mean,cv_accuracy_mean,model_complexity
0,2006-07,forgets,0.71478,0.34123,0.85036,0.68575,0.35934,0.82834,1
1,2006-07,basic,0.66739,0.34620,0.84951,0.63804,0.36269,0.82809,0
2,2006-09,forgets,0.71925,0.31465,0.87456,0.68430,0.33571,0.84836,1
3,2006-09,basic,0.66891,0.31975,0.87456,0.65578,0.33926,0.84836,0
4,2008-09,forgets,0.71902,0.30183,0.88724,0.67932,0.32371,0.85909,1
5,2008-09,basic,0.67526,0.30602,0.88724,0.64035,0.32817,0.85909,0


Saved Excel file: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/data/outputs/bkt_model_evaluations_v.0.xlsx


- dataset_name: the dataset used to train and evaluate the model. In your table, each row is a different dataset version.
- model_name: the BKT variant selected as best for that dataset. Here, all three best models are forgets, meaning the version that includes a forgetting parameter.
train_auc: AUC on the training data. It tells you how well the model separates correct vs incorrect responses on data it has already seen.
train_rmse: RMSE on the training data. It measures how close the predicted probabilities are to the observed outcomes. Lower is better.
train_accuracy: accuracy on the training data. It is the percentage of responses correctly classified as right or wrong.
cv_auc_mean: mean AUC across cross-validation folds. This is usually the most important column for model selection because it reflects performance on unseen data.
cv_rmse_mean: mean RMSE across cross-validation folds. Lower is better, and it tells you how good the probability estimates are on unseen data.
cv_accuracy_mean: mean accuracy across cross-validation folds. It shows classification performance on unseen data, but it can be less informative if the data are imbalanced.

## 11. Select the overall best dataset–model combination

After selecting the best model **within each dataset**, choose the **overall best candidate** across datasets using the same criteria:

1. Highest `cv_auc_mean`
2. Lowest `cv_rmse_mean`
3. If tied, choose the simpler model

In [26]:
overall_best = (
    best_models.sort_values(
        by=["cv_auc_mean", "cv_rmse_mean", "model_complexity"],
        ascending=[False, True, True]
    )
    .iloc[0]
)

best_dataset_name = overall_best["dataset_name"]
best_model_name = overall_best["model_name"]

print("OVERALL BEST MODEL")
display(overall_best[[
    "dataset_name",
    "model_name",
    "train_auc",
    "train_rmse",
    "train_accuracy",
    "cv_auc_mean",
    "cv_rmse_mean",
    "cv_accuracy_mean",
    "model_complexity"
]])


OVERALL BEST MODEL


dataset_name        2006-07
model_name          forgets
train_auc           0.71478
train_rmse          0.34123
train_accuracy      0.85036
cv_auc_mean         0.68575
cv_rmse_mean        0.35934
cv_accuracy_mean    0.82834
model_complexity          1
Name: 0, dtype: object

## 12. Save the KC probability table for the selected model

The following code refits the selected dataset–model combination, extracts the learned BKT probabilities (`prior`, `learns`, `guesses`, `slips`, and `forgets`) for each KC, and saves them to Excel.


In [27]:
chosen_dataset = best_dataset_name
chosen_model_name = best_model_name

chosen_df = datasets_bkt[chosen_dataset].copy()
chosen_model_spec = MODEL_SPECS[chosen_model_name]

chosen_defaults = BASE_DEFAULTS.copy()
chosen_defaults.update(chosen_model_spec["defaults_extra"])

chosen_fit_kwargs = chosen_model_spec["fit_kwargs"].copy()

chosen_model = Model(seed=42, num_fits=1)
chosen_model.fit(
    data=chosen_df,
    defaults=chosen_defaults,
    **chosen_fit_kwargs
)

chosen_params_long = normalise_bkt_params(
    raw_params=chosen_model.params(),
    dataset_name=chosen_dataset,
    model_name=chosen_model_name
)

chosen_params_wide = make_kc_probability_table(chosen_params_long)

chosen_train_predictions = chosen_model.predict(data=chosen_df)
if isinstance(chosen_train_predictions, pd.DataFrame):
    chosen_train_predictions = chosen_train_predictions.reset_index(drop=True).copy()
    if "dataset_name" not in chosen_train_predictions.columns:
        chosen_train_predictions.insert(0, "dataset_name", chosen_dataset)
    if "model_name" not in chosen_train_predictions.columns:
        chosen_train_predictions.insert(1, "model_name", chosen_model_name)

print("KC probabilities for chosen model")
display(chosen_params_wide)

probability_sheets = {
    "kc_probabilities_wide": chosen_params_wide,
    "chosen_model_metrics": overall_best.to_frame().T,
}


save_sheets_to_excel(CHOSEN_MODEL_PROBABILITIES_XLSX, probability_sheets)


KC probabilities for chosen model


parameter,dataset_name,model_name,skill_name,prior,learns,guesses,slips,forgets
0,2006-07,forgets,combine_like_terms,0.86262,0.08001,0.60444,0.05528,0.02251
1,2006-07,forgets,expand_eliminate_parentheses,0.83946,0.15964,0.50242,0.05256,0.03078
2,2006-07,forgets,move_constants,0.23268,0.08303,0.36890,0.09899,0.01487
3,2006-07,forgets,normalize_negative_sign,0.58617,0.35732,0.41474,0.09449,0.10694
4,2006-07,forgets,remove_coefficient,0.37680,0.07166,0.49771,0.08364,0.00975


Saved Excel file: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/data/outputs/bkt_chosen_model_probabilities_v.0.xlsx


In [28]:
print("Saved outputs:")
print("- Model evaluations:", MODEL_EVALUATIONS_XLSX)
print("- Chosen model probabilities:", CHOSEN_MODEL_PROBABILITIES_XLSX)


Saved outputs:
- Model evaluations: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/data/outputs/bkt_model_evaluations_v.0.xlsx
- Chosen model probabilities: /mnt/c/Users/Melany Nuria/Desktop/TFG/adaptive_bayesian_its/backend_fastAPI/app/data/outputs/bkt_chosen_model_probabilities_v.0.xlsx


The notebook now exports two Excel workbooks under `data/outputs/`:

- `bkt_model_evaluations.xlsx`: model metrics, ranking, best model per dataset, and learned parameters for all tested models.
- `bkt_chosen_model_probabilities.xlsx`: KC probability tables for the selected best model, plus the selected model metrics and train predictions when they fit within Excel's row limit.
